# Website Tech Stack Lookup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/website-tech-stack-detector.ipynb)

**What:** 2,392 technology fingerprints with version and confidence, tested on 19,393 checks. Find the CMS, ecommerce platform, frameworks, analytics, CDN and hosting any website uses. Bulk domains in, one row per site with evidence. $0.002 per site, no subscription; failed sites are free.

**Why:** run the [`website-tech-stack-detector`](https://apify.com/transparent_meteorite/website-tech-stack-detector) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/website-tech-stack-detector) for every option.

In [2]:
run_input = {'urls': ['shopify.com', 'wordpress.org', 'vercel.com']}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/website-tech-stack-detector').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> Status: RUNNING, Message: Starting Website Tech Stack Detector: reading input


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:57.992Z ACTOR: Pulling container image of build VXJz6grKmc6rNrExp from registry.


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:57.995Z ACTOR: Creating container.


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:58.065Z ACTOR: Starting container.


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:58.067Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:59.651Z [tech-stack] done: 3 analyzed, 0 not billable (failed/blocked/robots), 0 not attempted


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:59.722Z INFO  [Status message]: Done: 3 items


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> 2026-10-13T22:20:59.775Z Thanks for using Website Tech Stack Detector. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/nm3csrn1JbV3Vbgzo/issues) with the run URL and we will reply there.


[apify.website-tech-stack-detector runId:sTzkYNgU8wWiNApFo] -> Status: SUCCEEDED, Message: Done: 3 items


Run status: SUCCEEDED | items: 3


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)
df['n_technologies'] = df['technologies'].apply(lambda t: len(t) if isinstance(t, list) else 0)
df['categories'] = df['categoriesSummary'].apply(lambda c: ', '.join(sorted(c)) if isinstance(c, dict) else '')
TECH = df['technologies'].copy()

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["url", "status", "title", "server", "n_technologies", "categories"] if c in df.columns]
df[SHOW]

,url,status,title,server,n_technologies,categories
0,https://shopify.com/,200,Shopify: The All-in-One Commerce Platform for Businesses - Shopify,cloudflare,9,"CDN, Ecommerce, Metadata, Performance, SEO, Security, UI frameworks"
1,https://wordpress.org/,200,"Blog Tool, Publishing Platform, and CMS &#8211; WordPress.org",nginx,14,"Analytics, CMS, Fonts, Metadata, Miscellaneous, Page builders, Pro..."
2,https://vercel.com/,200,Agentic Infrastructure - Vercel,Vercel,10,"Hosting, JavaScript frameworks, Metadata, Miscellaneous, SEO, Secu..."


## 5. A quick analysis

In [5]:
from collections import Counter
cnt=Counter(t['name'] for ts in TECH if isinstance(ts,list) for t in ts)
print('Most common technologies:',cnt.most_common(8))

Most common technologies: [('HSTS', 3), ('Open Graph', 3), ('Schema.org JSON-LD', 3), ('Twitter Cards', 3), ('Choices', 2), ('Cloudflare', 1), ('Tailwind CSS', 1), ('HTTP/3', 1)]


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [website-tech-stack-detector on the Apify Store](https://apify.com/transparent_meteorite/website-tech-stack-detector).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.